# Fine-tune NCM-iWSP-GPT1 (new notebook)

This notebook gradient-fine-tunes the **untied** NCM-iWSP-GPT1 generator from the converted v10 init. It does not train NCM-GPT1, and it does not edit the kernel `nicholashuihk/ncm-gpt-1-train`.

Attach three inputs, all read-only:

1. This repository.
2. A **new** dataset uploaded from `E:\\Project\\ncm-iwsp-gpt1-init` with `tokenizer.pt`, `generator.pt`, and `convert_report.json`. Do not write those files into `E:\\Project\\ncm-gpt1-kaggle-100k` or into a v10 kernel output.
3. The existing v10 export that contains `artifacts/packed_windows.pt` (BookCorpus Open, GPT-2 BPE, context 512). The script only reads that file.

Outputs go to `/kaggle/working/finetune`. Accelerator: **GPU T4 x2**. Do not reinstall PyTorch.

**Nicholas confirms before any submit.** This notebook does not call the Kaggle API. `NICHOLAS_CONFIRMED` below is `False`, so Run All stops before the first training step. Set it to `True` only after the printed paths are the converted init and `artifacts/packed_windows.pt`.

Defaults: batch 1 per GPU, gradient checkpointing, chunked block attention, fp16 on T4, alpha 0, mid_aux 0. `val_ce_first`, `val_ce_mid`, and `val_ce_last` are raw level means. A loss that includes `mid_aux` is not comparable to an alpha=0 curve. See `docs/finetune_kaggle.md`.


In [ ]:
# Nicholas sets this to True after checking the paths printed below.
# Leave it False until then. This notebook does not submit a Kaggle job.
NICHOLAS_CONFIRMED = False
print("NICHOLAS_CONFIRMED", NICHOLAS_CONFIRMED)


In [ ]:
import sys
from pathlib import Path

ROOT = None
here = Path.cwd()
candidates = [here, here.parent]
kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    candidates.extend(path for path in kaggle_input.iterdir() if path.is_dir())
for cand in candidates:
    if (cand / "ncm_iwsp_gpt1" / "finetune.py").is_file():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("Attach this repository as a Kaggle input. Do not clone into the v10 kernel.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo", ROOT)

from ncm_iwsp_gpt1.finetune import find_converted_pair, find_packed_windows

tokenizer, generator = find_converted_pair(kaggle_input)
packed = find_packed_windows(kaggle_input)
print("tokenizer", tokenizer)
print("generator", generator)
print("packed", packed)
print("out", "/kaggle/working/finetune")


In [ ]:
import torch
print("cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(index, props.name, "mem_gib", round(props.total_memory / 1024**3, 1))


The next cell is the training run. It writes only under `/kaggle/working/finetune`. Per-GPU batch is 1, with checkpointing and chunked attention. A later session resumes by attaching this output as a read-only dataset and passing `--resume` at that `latest.pt`.


In [ ]:
import subprocess
import sys
from pathlib import Path

if not NICHOLAS_CONFIRMED:
    raise SystemExit(
        "Nicholas must set NICHOLAS_CONFIRMED = True after reviewing the paths. "
        "This notebook does not submit a Kaggle job."
    )

script = Path(ROOT) / "kaggle" / "finetune_iwsp.py"
command = [
    sys.executable,
    str(script),
    "--confirm",
    "--config", str(Path(ROOT) / "configs" / "target.yaml"),
    "--tokenizer", str(tokenizer),
    "--generator", str(generator),
    "--packed", str(packed),
    "--out", "/kaggle/working/finetune",
    "--device", "cuda",
    "--batch-size", "1",
    "--max-hours", "11.5",
    "--reserve-minutes", "15",
]
print("running", " ".join(command), flush=True)
raise SystemExit(subprocess.call(command))


In [ ]:
import json
from pathlib import Path

out = Path("/kaggle/working/finetune")
report = out / "report.json"
status = out / "status.json"
val_log = out / "val_log.jsonl"
print("report", report.read_text(encoding="utf-8") if report.is_file() else "missing")
print("status", status.read_text(encoding="utf-8") if status.is_file() else "missing")
if val_log.is_file():
    lines = val_log.read_text(encoding="utf-8").splitlines()
    print("val rows", len(lines))
    if lines:
        last = json.loads(lines[-1])
        print("val_ce_first", last.get("val_ce_first"))
        print("val_ce_mid", last.get("val_ce_mid"))
        print("val_ce_last", last.get("val_ce_last"))
        print("loss_comparable_to_alpha0", last.get("loss_comparable_to_alpha0"))
        print(last.get("loss_note"))
